1.-Carga y creación 

In [ ]:
import sys; sys.path.append("..")
import pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src.datos import cargar_obt
from src import pipeline as P

df = P.agregar_features(cargar_obt())
df[["fecha_coccion", "coccion_mes", "carga_planta_14d", "temporada_alta", "generacion_levadura", "levadura_nueva"]].head(10)

2.- Importancia de features 

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
df["carga_q"] = pd.qcut(df["carga_planta_14d"], 4, duplicates="drop")
sns.barplot(data=df, x="carga_q", y="merma_alta", ax=ax[0], errorbar=None); ax[0].set_title("% merma alta por carga de planta (14 d)")
sns.barplot(data=df, x="temporada_alta", y="merma_alta", ax=ax[1], errorbar=None); ax[1].set_title("% merma alta por temporada")
sns.barplot(data=df, x="levadura_nueva", y="merma_alta", ax=ax[2], errorbar=None); ax[2].set_title("% merma alta: levadura nueva")
for a in ax: a.set_ylim(0, 1); a.set_ylabel("proporción merma alta")
plt.tight_layout(); plt.savefig("../docs/figs/05_features.png", dpi=150); plt.show()

print(df[["carga_planta_14d", "temporada_alta", "levadura_nueva", "merma_pct"]]
      .corr(method="spearman")["merma_pct"].round(3))

3.- Prueba de que no hay fuga de datos 

In [ ]:
FUGA = ["cantidad_completada", "merma_litros", "merma_ajustada", "fecha_cierre", "dias_proceso"] \
       + [c for c in df.columns if c.startswith("cierre_")] + ["merma_pct"]
assert not set(P.FEATURES) & set(FUGA), "¡Hay una feature con fuga!"
print("Features del modelo:", P.FEATURES)
print("Ninguna feature usa información posterior a la cocción ✔")

# La carga de un lote solo cuenta cocciones ANTERIORES: ejemplo con un lote cualquiera
i = 100
f = df.loc[i, "fecha_coccion"]
manual = ((df["fecha_coccion"] >= f - pd.Timedelta(days=14)) & (df["fecha_coccion"] < f)).sum()
print(f"Lote {df.loc[i,'numero_documento']} ({f.date()}): carga calculada={df.loc[i,'carga_planta_14d']}, recuento manual={manual}")

4.- División y validación cruzada 

In [ ]:
train, test = P.dividir_train_test(df)
print(f"Train: {len(train)} lotes  {train.fecha_coccion.min().date()} → {train.fecha_coccion.max().date()} | % alta {train.merma_alta.mean():.0%}")
print(f"Test:  {len(test)} lotes  {test.fecha_coccion.min().date()} → {test.fecha_coccion.max().date()} | % alta {test.merma_alta.mean():.0%}")

for k, (i_tr, i_va) in enumerate(P.esquema_cv().split(train), 1):
    tr, va = train.iloc[i_tr], train.iloc[i_va]
    print(f"Pliegue {k}: entrena hasta {tr.fecha_coccion.max().date()} ({len(tr)}) | valida {va.fecha_coccion.min().date()}→{va.fecha_coccion.max().date()} ({len(va)}) | % alta val {va.merma_alta.mean():.0%}")

5.- Funcionamiento del pipeline 

In [ ]:
from sklearn.dummy import DummyClassifier
X_tr, y_tr = P.preparar_X_y(train)
pipe = P.construir_pipeline(DummyClassifier()).fit(X_tr, y_tr)
print("Columnas después de la preparación:")
print(list(pipe.named_steps["prep"].get_feature_names_out()))